# Full SqueezeLLM Pipeline for Llama 3.2 1B Instruct

This notebook runs the full dense-only SqueezeLLM pipeline for `meta-llama/Llama-3.2-1B-Instruct` from your fork:
`https://github.com/thangSy221105/SqueezeLLM.git`

Pipeline:
- install repo and CUDA extension
- login to Hugging Face
- collect gradient-square statistics
- chunk model weights
- run NUQ quantization
- pack a SqueezeLLM checkpoint
- evaluate the packed checkpoint

Important:
- Use a GPU runtime in Colab.
- You must have access to `meta-llama/Llama-3.2-1B-Instruct` on Hugging Face.
- Gradient collection and KMeans quantization are the expensive steps.


In [ ]:
MODEL_ID = 'meta-llama/Llama-3.2-1B-Instruct'
WBITS = 4
NSAMPLES = 128
SEQLEN = 2048
DATASET = 'c4'

# Paper-like D+S defaults: 0.05% sensitive values plus outliers from the generated threshold config.
OUTLIER_RANGE = 1.8
SENSITIVITY = 0.05

REPO_URL = 'https://github.com/thangSy221105/SqueezeLLM.git'
REPO_DIR = '/content/SqueezeLLM'

GRAD_DIR = '/content/grad_chunks_llama32_1b'
MODEL_CHUNK_DIR = '/content/model_chunks_llama32_1b'
LUT_DIR = f'/content/lut_llama32_1b_w{WBITS}'
PACKED_CKPT = f'/content/llama32_1b_instruct_w{WBITS}.pt'

OUTLIER_DIR = '/content/outlier_cfg_llama32_1b'
LUT_DIR_DS = f'/content/lut_llama32_1b_w{WBITS}_ds'
PACKED_CKPT_DS = f'/content/llama32_1b_instruct_w{WBITS}_ds.pt'


In [ ]:
!nvidia-smi

!pip -q install -U transformers accelerate sentencepiece huggingface_hub datasets scikit-learn

%cd /content
!rm -rf /content/SqueezeLLM
!git clone {REPO_URL}
%cd {REPO_DIR}
!git rev-parse --short HEAD


## Hugging Face login

Preferred setup:
- In Colab Secrets, create `HF_TOKEN` once.
- The next cell tries Colab Secrets first, then `HF_TOKEN` in the environment, then falls back to manual login.


In [ ]:
import os
from huggingface_hub import login

hf_token = None

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print('Logged in using stored HF_TOKEN')
else:
    print('HF_TOKEN not found; falling back to manual login prompt')
    login()


In [ ]:
%cd {REPO_DIR}
!pip -q install -e .
%cd {REPO_DIR}/squeezellm
!python setup_cuda.py install
%cd {REPO_DIR}


In [ ]:
import torch
import transformers

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## Optional: verify the base model first

This is not part of quantization, but it is useful to confirm HF access and runtime health before spending time on gradient collection.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map='auto'
)
model.eval()

messages = [
    {'role': 'system', 'content': 'You are a concise technical assistant.'},
    {'role': 'user', 'content': 'Explain quantization in three short sentences.'},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt'
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128)

prompt_len = inputs['input_ids'].shape[-1]
print(tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True))


## Quantization pipeline

The notebook now contains two branches:
- `dense-only`: shorter path for a quick first packed checkpoint
- `dense-and-sparse`: closer to the paper, using outlier extraction plus `--sensitivity 0.05`

If you want the closest path to the paper, focus on the dense-and-sparse cells below.


In [ ]:
%cd {REPO_DIR}/quantization
!python collect_gradients.py \
  --model {MODEL_ID} \
  --dataset {DATASET} \
  --nsamples {NSAMPLES} \
  --seqlen {SEQLEN} \
  --output {GRAD_DIR}


In [ ]:
%cd {REPO_DIR}/quantization
!python chunk_models.py \
  --model {MODEL_ID} \
  --model_type llama \
  --output {MODEL_CHUNK_DIR}


In [ ]:
%cd {REPO_DIR}/quantization
!python nuq.py \
  --bit {WBITS} \
  --model_type llama \
  --model {MODEL_CHUNK_DIR} \
  --gradient {GRAD_DIR} \
  --output {LUT_DIR}


In [ ]:
%cd {REPO_DIR}/quantization
!python pack.py \
  --model {MODEL_ID} \
  --wbits {WBITS} \
  --folder {LUT_DIR} \
  --save {PACKED_CKPT}


In [ ]:
%cd {REPO_DIR}
!python llama.py {MODEL_ID} c4 \
  --wbits {WBITS} \
  --load {PACKED_CKPT} \
  --eval


## Dense-and-sparse branch (closest to the paper)

This branch adds outlier extraction and sparse packing. It is the better match to the original SqueezeLLM paper than the dense-only branch above.

Paper-like defaults used here:
- `OUTLIER_RANGE = 1.8` as a practical starting point to generate the outlier threshold file
- `SENSITIVITY = 0.05` to extract an additional 0.05% of sensitive values

You should inspect the generated outlier percentage in the output filename and logs. If it is far from the paper target, tune `OUTLIER_RANGE`.


In [ ]:
%cd {REPO_DIR}/quantization
!python generate_outlier_config.py \
  --model {MODEL_CHUNK_DIR} \
  --model_type llama \
  --range {OUTLIER_RANGE} \
  --output {OUTLIER_DIR}


In [ ]:
import glob

matches = sorted(glob.glob(f'{OUTLIER_DIR}/outlier_config_o*.json'))
assert matches, 'No outlier config file found'
OUTLIER_CONFIG_PATH = matches[-1]
print('Using outlier config:', OUTLIER_CONFIG_PATH)


In [ ]:
%cd {REPO_DIR}/quantization
!python nuq.py \
  --bit {WBITS} \
  --model_type llama \
  --model {MODEL_CHUNK_DIR} \
  --gradient {GRAD_DIR} \
  --output {LUT_DIR_DS} \
  --outlier_config {OUTLIER_CONFIG_PATH} \
  --sensitivity {SENSITIVITY}


In [ ]:
%cd {REPO_DIR}/quantization
!python pack.py \
  --model {MODEL_ID} \
  --wbits {WBITS} \
  --folder {LUT_DIR_DS} \
  --save {PACKED_CKPT_DS} \
  --include_sparse \
  --balance


In [ ]:
%cd {REPO_DIR}
!python llama.py {MODEL_ID} c4 \
  --wbits {WBITS} \
  --load {PACKED_CKPT_DS} \
  --include_sparse \
  --eval


## Output artifacts

Expected paths after a successful run:
- `GRAD_DIR`: layer-wise gradient-square chunks
- `MODEL_CHUNK_DIR`: layer-wise model chunks
- `LUT_DIR`: LUT files for the dense-only branch
- `PACKED_CKPT`: packed dense-only checkpoint
- `OUTLIER_DIR`: generated outlier threshold config
- `LUT_DIR_DS`: LUT plus sparse outlier files for the dense-and-sparse branch
- `PACKED_CKPT_DS`: packed dense-and-sparse checkpoint


In [ ]:
!ls -lah /content | sed -n '1,120p'
!ls -lah {LUT_DIR} | sed -n '1,120p'


## Notes

- This notebook is written for the fork at `thangSy221105/SqueezeLLM`.
- It uses the in-repo gradient collection path added to your fork.
- The dense-and-sparse branch is the better match to the paper.
- The exact outlier percentage depends on the generated threshold config, so you may need to tune `OUTLIER_RANGE`.
- If Colab RAM or GPU time is tight, reduce `NSAMPLES` first.
